# Actividad · Secuencias frecuentes de navegación web

Una tienda virtual dispone de 12 sesiones ficticias. Cada sesión es una secuencia ordenada de páginas y puede repetir páginas. Se quiere descubrir recorridos frecuentes que comiencen en `inicio` y terminen en `producto`.

1. **Explorar los datos.** Distinguir conjunto de páginas, secuencia y subsecuencia. Calcular manualmente el soporte de `(inicio, producto)` y `(producto, inicio)` sin restricciones de separación.
2. **Cobertura ordenada.** Implementar una función que compruebe si un patrón aparece como subsecuencia. Una sesión cuenta una sola vez aunque contenga varias apariciones. Permitir un parámetro `max_gap`: cantidad máxima de eventos omitidos entre dos páginas consecutivas del patrón. `None` significa sin límite; `0`, eventos contiguos.
3. **Minería con restricciones.** Con OR-Tools, enumerar patrones de longitud 2 a 4, que comiencen en inicio, terminen en producto y aparezcan en al menos 50 % de las sesiones. Resolver primero sin límite de separación y luego con `max_gap=1`.
4. **Validación e interpretación.** Comparar con una búsqueda exhaustiva independiente, reportar soporte y sesiones cubiertas e interpretar dos patrones.
5. **Sensibilidad.** Comparar los soportes mínimos 25 %, 50 % y 75 % con `max_gap=None`, `1` y `0`. Explicar por qué aumentar el soporte o reducir la separación no puede agregar patrones, manteniendo las demás condiciones.

**Entrega:** este notebook completado, con código ejecutado, tablas y conclusiones. Los datos se incluyen abajo. No se requiere implementar PrefixSpan ni medir tiempos.


In [ ]:
import importlib.util, subprocess, sys
faltantes = [p for p in ["pandas", "ortools", "matplotlib"] if importlib.util.find_spec(p) is None]
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltantes])
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ortools.sat.python import cp_model
from itertools import combinations, product
from importlib.metadata import version
print({p: version(p) for p in ["pandas", "ortools", "matplotlib"]})

## 1. Sesiones y soporte
El soporte es la proporción de sesiones que contienen el patrón en el orden indicado; no es el número total de apariciones.

In [ ]:
sesiones = [
 ["inicio", "categoria", "producto", "carrito", "pago"],
 ["inicio", "busqueda", "producto", "carrito"],
 ["inicio", "categoria", "producto", "producto", "carrito"],
 ["inicio", "busqueda", "categoria", "producto"],
 ["inicio", "categoria", "busqueda", "producto"],
 ["inicio", "producto", "carrito", "pago"],
 ["inicio", "busqueda", "producto", "inicio", "producto"],
 ["inicio", "categoria", "producto"],
 ["inicio", "busqueda", "categoria", "busqueda", "producto"],
 ["inicio", "categoria", "carrito"],
 ["inicio", "busqueda", "producto", "carrito", "producto"],
 ["inicio", "categoria", "producto", "inicio", "busqueda", "producto"]
]
N = len(sesiones)
alfabeto = sorted({p for s in sesiones for p in s})
codigo = {p:i for i,p in enumerate(alfabeto)}
longitudes = range(2, 5)
display(pd.DataFrame({"sesion": range(1, N+1), "recorrido": [" → ".join(s) for s in sesiones]}))

## 2. Cobertura con separación máxima
Para posiciones elegidas $j_1<\cdots<j_L$, se requiere que los eventos coincidan con el patrón. La separación es $j_{r+1}-j_r-1$.

Usamos búsqueda con retroceso: con límites de separación, tomar siempre la primera aparición de una página puede descartar una aparición posterior válida. La función se detiene cuando encuentra una cobertura completa.

In [ ]:
def cubre(sesion, patron, max_gap=None):
    if max_gap is not None and max_gap < 0:
        raise ValueError("max_gap debe ser no negativo o None")
    def buscar(k, anterior):
        if k == len(patron):
            return True
        fin = len(sesion) if anterior < 0 or max_gap is None else min(len(sesion), anterior + max_gap + 2)
        return any(sesion[j] == patron[k] and buscar(k+1, j)
                   for j in range(anterior+1, fin))
    return buscar(0, -1)

def frecuencia(patron, gap=None):
    return sum(cubre(s, patron, gap) for s in sesiones)

assert cubre(["a", "x", "a", "b"], ("a", "b"), 0)
assert not cubre(["a", "x", "b"], ("a", "b"), 0)
assert cubre(["a", "x", "b"], ("a", "b"), 1)
assert not cubre(["a"], ("a", "a"))
assert frecuencia(("inicio", "producto")) == 11
assert frecuencia(("producto", "inicio")) == 2
for p in [("inicio", "producto"), ("producto", "inicio")]:
    print(p, frecuencia(p), frecuencia(p)/N)

**Respuesta manual:** `(inicio, producto)` cubre 11/12 sesiones (91.67 %); `(producto, inicio)` cubre 2/12 (16.67 %). El orden cambia el resultado. Repetir un evento en un patrón requiere posiciones diferentes de la sesión.

## 3. Modelo de restricciones
Para cada longitud fija $L$, se definen variables enteras $P_1,\ldots,P_L$ que identifican páginas y variables booleanas $T_s$ para cobertura. Una tabla permitida por sesión relaciona la tupla completa del patrón con su indicador exacto de cobertura.

Se imponen $P_1=\text{inicio}$, $P_L=\text{producto}$ y $\sum_s T_s\geq\lceil N\cdot minsup\rceil$. No hay objetivo: se enumeran soluciones factibles.

Las tablas se construyen a partir de todas las combinaciones de posiciones de cada sesión, de manera independiente de la función de retroceso. Se incluyen filas tanto con cobertura 1 como con cobertura 0, evitando una implicación incompleta. El costo de las tablas crece como $|alfabeto|^L$; por eso limitamos la longitud a cuatro.

In [ ]:
from functools import lru_cache
from fractions import Fraction
from math import ceil

@lru_cache(None)
def preparar_tablas(L, gap):
    universo = list(product(range(len(alfabeto)), repeat=L))
    tablas = []
    for s in sesiones:
        presentes = set()
        for posiciones in combinations(range(len(s)), L):
            if gap is None or all(b-a-1 <= gap for a,b in zip(posiciones, posiciones[1:])):
                presentes.add(tuple(codigo[s[j]] for j in posiciones))
        tablas.append([list(p) + [int(p in presentes)] for p in universo])
    return tablas

class Patrones(cp_model.CpSolverSolutionCallback):
    def __init__(self, P, T):
        super().__init__()
        self.P, self.T, self.resultados = P, T, {}
    def on_solution_callback(self):
        patron = tuple(alfabeto[self.value(v)] for v in self.P)
        self.resultados[patron] = sum(self.value(v) for v in self.T)

def minar_cp(soporte=0.5, gap=None):
    resultados = {}
    for L in longitudes:
        modelo = cp_model.CpModel()
        P = [modelo.new_int_var(0, len(alfabeto)-1, f"P_{r}") for r in range(L)]
        T = [modelo.new_bool_var(f"T_{s}") for s in range(N)]
        for s, tabla in enumerate(preparar_tablas(L, gap)):
            modelo.add_allowed_assignments(P + [T[s]], tabla)
        modelo.add(P[0] == codigo["inicio"])
        modelo.add(P[-1] == codigo["producto"])
        modelo.add(sum(T) >= ceil(Fraction(str(soporte))*N))
        solver = cp_model.CpSolver()
        solver.parameters.enumerate_all_solutions = True
        solver.parameters.num_search_workers = 1
        recolector = Patrones(P, T)
        estado = solver.solve(modelo, recolector)
        assert estado in (cp_model.OPTIMAL, cp_model.INFEASIBLE), solver.status_name(estado)
        resultados.update(recolector.resultados)
    return resultados

def presentar(resultados, gap=None):
    filas = [{"patron": " → ".join(p), "longitud":len(p), "frecuencia":f,
              "soporte":f/N, "sesiones":[i+1 for i,s in enumerate(sesiones) if cubre(s,p,gap)]}
             for p,f in resultados.items()]
    return pd.DataFrame(filas, columns=["patron","longitud","frecuencia","soporte","sesiones"]).sort_values(
        ["frecuencia","patron"], ascending=[False,True]).reset_index(drop=True)

sin_gap = minar_cp()
con_gap = minar_cp(gap=1)
print("Sin límite de separación")
display(presentar(sin_gap))
print("Como máximo un evento omitido entre páginas del patrón")
display(presentar(con_gap, 1))

## 4. Validación y sensibilidad
La referencia enumera todas las tuplas, filtra los extremos y calcula la cobertura con retroceso. Así contrastamos dos implementaciones de cobertura y la búsqueda de CP-SAT.

In [ ]:
def minar_referencia(soporte, gap):
    resultados = {}
    for L in longitudes:
        for p in product(alfabeto, repeat=L):
            if p[0] == "inicio" and p[-1] == "producto":
                f = frecuencia(p, gap)
                if f >= ceil(Fraction(str(soporte))*N):
                    resultados[p] = f
    return resultados

filas = []
for gap in [None, 1, 0]:
    anterior = None
    for soporte in [0.25, 0.5, 0.75]:
        resultado = minar_cp(soporte, gap)
        assert resultado == minar_referencia(soporte, gap)
        if anterior is not None:
            assert set(resultado) <= anterior
        anterior = set(resultado)
        filas.append({"max_gap": "sin limite" if gap is None else str(gap),
                      "soporte_minimo":soporte, "patrones":len(resultado), "validado":True})
assert set(con_gap) <= set(sin_gap)
display(pd.DataFrame(filas))
for p in [("inicio","producto"), ("inicio","categoria","producto")]:
    print(" → ".join(p), {str(g): frecuencia(p,g) for g in [None,1,0]})
print("Todas las verificaciones de secuencias pasaron.")

## 5. Interpretación y conclusiones
- `(inicio, producto)` expresa que producto se visitó después de inicio, sin exigir que sean contiguos ni que la sesión termine allí. Sin límite de separación aparece en 11 de 12 sesiones.
- `(inicio, categoria, producto)` describe el paso ordenado por categoría antes del producto; aparece en 7 de 12 sesiones sin límite de separación. No exige que esas sean las únicas páginas visitadas.
- Los límites se aplican al **patrón**, no al primer y último evento de toda la sesión. `max_gap` cuenta eventos omitidos, no segundos.
- La frecuencia cuenta sesiones y no apariciones. Cada $T_s$ es booleano incluso si hay múltiples formas de encontrar el patrón.
- Subir el soporte mínimo o reducir `max_gap` mantiene o reduce los patrones. La tabla permite observar cuándo desaparecen.
- Una secuencia frecuente no demuestra que un recorrido cause compras ni mide conversión. Para ello harían falta eventos de compra, contexto y un diseño de evaluación.

**Extensión opcional:** añadir un límite al número total de eventos entre la primera y última posición del patrón (*max-span*), distinguiéndolo de `max_gap`.

**Referencia técnica:** [OR-Tools CP-SAT](https://developers.google.com/optimization/cp/cp_solver). Este modelo de tablas es una construcción didáctica propia, no una implementación de PrefixSpan.